### 1. Download/Load SP500 stocks prices data.

In [2]:
from statsmodels.regression.rolling import RollingOLS
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pandas as pd
import numpy as np
import datetime as dt
import yfinance as yf
import pandas_ta
import warnings
warnings.filterwarnings('ignore')


In [3]:
import pandas as pd
import requests
import certifi
from io import StringIO

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, verify=certifi.where())
html = response.text

tables = pd.read_html(StringIO(html))

sp500 = tables[0]

sp500['Symbol'] = sp500["Symbol"].str.replace('.', '-')

symbols_list = sp500["Symbol"].unique().tolist()

end_date = '2026-03-20'

start_date = pd.to_datetime(end_date)-pd.DateOffset(365*8)

df = yf.download(
    tickers=symbols_list,
    start=start_date,
    end=end_date,
    auto_adjust=False  # 👈 IMPORTANT
).stack()

df.index.names = ['date', 'ticker']

df.columns= df.columns.str.lower()

df

[*********************100%***********************]  503 of 503 completed


Price               adj close       close        high         low        open  \
date       ticker                                                               
2018-03-22 A        63.545284   67.470001   69.199997   67.330002   68.699997   
           AAPL     39.667385   42.212502   43.169998   42.150002   42.500000   
           ABBV     69.621117   98.099998  104.500000   95.519997  104.190002   
           ABNB           NaN         NaN         NaN         NaN         NaN   
           ABT      52.924831   60.930000   62.099998   60.849998   61.820000   
...                       ...         ...         ...         ...         ...   
2026-03-19 XYZ      58.990002   58.990002   59.387001   57.119999   57.240002   
           YUM     156.250000  156.250000  159.259995  156.250000  157.910004   
           ZBH      89.839996   89.839996   90.760002   88.620003   88.620003   
           ZBRA    206.190002  206.190002  208.710007  202.500000  205.000000   
           ZTS     115.989998  115.989998  118.360001  115.900002  116.690002   

Price                   volume  
date       ticker               
2018-03-22 A         1690000.0  
           AAPL    165963200.0  
           ABBV     26861200.0  
           ABNB            NaN  
           ABT       5345300.0  
...                        ...  
2026-03-19 XYZ       6387300.0  
           YUM       1832100.0  
           ZBH       1812000.0  
           ZBRA       593300.0  
           ZTS       4999100.0  

[1010527 rows x 6 columns]

### 2. Calculate features and technical indicators for each stock.
- Garman-Klass Volatility
- RSI
- Bollinger Bands
- ATR
- MACD
- Dollar Volume

In [ ]:
# df['garman_klass_vol'] = ((np.log(df['high'])-np.log(df['low']))**2)/2-(2*np.log(2)-1)*(np.log(df['adj close'])-(np.log(df['open']))**2)

# df['rsi'] = df.groupby(level=1)['adj close'].transform(lambda x: pandas_ta.rsi(close=x, length=20))

# df['bb_low'] = df.groupby(level=1)['adj close'].transform(lambda x: pandas_ta.bbands(close=np.log1p(x), length=20).iloc[:,0])

# df['bb_mid'] = df.groupby(level=1)['adj close'].transform(lambda x: pandas_ta.bbands(close=np.log1p(x), length=20).iloc[:,1])

# df['bb_high'] = df.groupby(level=1)['adj close'].transform(lambda x: pandas_ta.bbands(close=np.log1p(x), length=20).iloc[:,2])

# def compute_atr(stock_data):
#     atr = pandas_ta.atr(high=stock_data['high'],
#                         low=stock_data['low'],
#                         close=stock_data['close'],
#                         length=14)
#     return atr.sub(atr.mean()).div(atr.std())

# df['atr'] = df.groupby(level=1, group_keys=False).apply(compute_atr)

# def compute_macd(close):
#     macd = pandas_ta.macd(close=close, length=20).iloc[:,0]
#     return macd.sub(macd.mean()).div(macd.std())

# df['macd'] = df.groupby(level=1, group_keys=False)['adj close'].apply(compute_macd)

# df['dollar_volume'] = (df['adj close']*df['volume'])/1e6

df



Price               adj close       close        high         low        open  \
date       ticker                                                               
2018-03-22 A        63.545284   67.470001   69.199997   67.330002   68.699997   
           AAPL     39.667385   42.212502   43.169998   42.150002   42.500000   
           ABBV     69.621117   98.099998  104.500000   95.519997  104.190002   
           ABNB           NaN         NaN         NaN         NaN         NaN   
           ABT      52.924831   60.930000   62.099998   60.849998   61.820000   
...                       ...         ...         ...         ...         ...   
2026-03-19 XYZ      58.990002   58.990002   59.387001   57.119999   57.240002   
           YUM     156.250000  156.250000  159.259995  156.250000  157.910004   
           ZBH      89.839996   89.839996   90.760002   88.620003   88.620003   
           ZBRA    206.190002  206.190002  208.710007  202.500000  205.000000   
           ZTS     115.989998  115.989998  118.360001  115.900002  116.690002   

Price                   volume  garman_klass_vol        rsi    bb_low  \
date       ticker                                                       
2018-03-22 A         1690000.0          5.307683        NaN       NaN   
           AAPL    165963200.0          4.009346        NaN       NaN   
           ABBV     26861200.0          6.704025        NaN       NaN   
           ABNB            NaN               NaN        NaN       NaN   
           ABT       5345300.0          5.037630        NaN       NaN   
...                        ...               ...        ...       ...   
2026-03-19 XYZ       6387300.0          4.753294  47.622919  3.927554   
           YUM       1832100.0          7.947278  45.700865  5.048697   
           ZBH       1812000.0          6.030893  42.287756  4.491567   
           ZBRA       593300.0          8.887404  37.090520  5.274545   
           ZTS       4999100.0          6.914710  39.486943  4.737676   

Price                bb_mid   bb_high       atr      macd  dollar_volume  
date       ticker                                                         
2018-03-22 A            NaN       NaN       NaN       NaN     107.391530  
           AAPL         NaN       NaN       NaN       NaN    6583.326167  
           ABBV         NaN       NaN       NaN       NaN    1870.106738  
           ABNB         NaN       NaN       NaN       NaN            NaN  
           ABT          NaN       NaN       NaN       NaN     282.899101  
...                     ...       ...       ...       ...            ...  
2026-03-19 XYZ     4.109759  4.291963 -0.541365  0.056086     376.786838  
           YUM     5.090827  5.132957  1.819296 -0.236914     286.265625  
           ZBH     4.567495  4.643423 -0.211920 -0.345068     162.790073  
           ZBRA    5.401648  5.528752  0.095328 -1.256195     122.332528  
           ZTS     4.821242  4.904809  0.126562 -1.013602     579.845598  

[1010527 rows x 14 columns]

### 3. Aggregate to monthly level and filter top 150 most liquid stocks for each month
- To reduce training time and experiment with features and strategies, we convert the business-daily data to month-end frequency

In [46]:
last_cols = [c for c in df.columns.unique(0) if c not in ['dollar_volume', 'volume', 'open', 'high', 'low', 'close']]

data = pd.concat([
    df.unstack('ticker')['dollar_volume'].resample('ME').mean().stack('ticker').to_frame('dollar_volume'),
    df.unstack()[last_cols].resample('ME').last().stack('ticker')
], axis=1).dropna(subset=['dollar_volume', 'adj close'])  # only drop if these are NaN

data

dollar_volume   adj close  garman_klass_vol        rsi  \
date       ticker                                                           
2018-03-31 A          143.276076   63.008434          5.217050   8.661129   
           AAPL      6347.313443   39.416023          3.974430  10.343383   
           ABBV       956.063361   67.172653          6.368071  14.217995   
           ABT        332.180042   52.047520          4.926442   7.529432   
           ACGL        64.065371   27.129135          3.075186  12.204337   
...                          ...         ...               ...        ...   
2026-03-31 XYZ        581.353228   58.990002          4.753294  47.622919   
           YUM        279.860158  156.250000          7.947278  45.700865   
           ZBH        200.072727   89.839996          6.030893  42.287756   
           ZBRA       162.709509  206.190002          8.887404  37.090520   
           ZTS        481.208041  115.989998          6.914710  39.486943   

                     bb_low    bb_mid   bb_high       atr      macd  
date       ticker                                                    
2018-03-31 A            NaN       NaN       NaN       NaN       NaN  
           AAPL         NaN       NaN       NaN       NaN       NaN  
           ABBV         NaN       NaN       NaN       NaN       NaN  
           ABT          NaN       NaN       NaN       NaN       NaN  
           ACGL         NaN       NaN       NaN       NaN       NaN  
...                     ...       ...       ...       ...       ...  
2026-03-31 XYZ     3.927554  4.109759  4.291963 -0.541365  0.056086  
           YUM     5.048697  5.090827  5.132957  1.819296 -0.236914  
           ZBH     4.491567  4.567495  4.643423 -0.211920 -0.345068  
           ZBRA    5.274545  5.401648  5.528752  0.095328 -1.256195  
           ZTS     4.737676  4.821242  4.904809  0.126562 -1.013602  

[47842 rows x 9 columns]

- Calculate 5-year rolling average of dollar volume for each stocks before filtering.

In [ ]:
# data['dollar_volume'] = (data['dollar_volume'].unstack('ticker').rolling(5*12).mean().stack('ticker')
#     .reindex(data.index))

# data['dollar_vol_rank'] = (data.groupby('date')['dollar_volume'].rank(ascending=False))

# data = data[data['dollar_vol_rank']<150].drop(['dollar_volume', 'dollar_vol_rank'], axis=1)

data

adj close  garman_klass_vol        rsi    bb_low  \
date       ticker                                                      
2023-02-28 AAPL    145.304962          7.698567  52.120135  4.963778   
           ABBV    138.085938          7.896076  54.426362  4.861898   
           ABT      95.904289          6.424245  36.762951  4.548466   
           ACN     254.677643          9.881769  41.630021  5.532744   
           ADBE    323.950012         10.667205  37.957137  5.774364   
...                       ...               ...        ...       ...   
2026-03-31 WDC     316.929993         10.229976  62.481774  5.485216   
           WFC      76.389999          5.556414  35.566775  4.293305   
           WMT     120.841995          7.075625  45.694634  4.804058   
           XOM     158.160004          7.951514  65.453187  4.986162   
           XYZ      58.990002          4.753294  47.622919  3.927554   

                     bb_mid   bb_high       atr      macd  
date       ticker                                          
2023-02-28 AAPL    5.006365  5.048952  0.056964  0.337159  
           ABBV    4.907300  4.952702 -0.156936 -0.065563  
           ABT     4.624205  4.699945  0.033138 -1.836136  
           ACN     5.596134  5.659525  0.218168 -0.658135  
           ADBE    5.896798  6.019232 -0.151346 -0.758460  
...                     ...       ...       ...       ...  
2026-03-31 WDC     5.622986  5.760756  6.106922  3.115020  
           WFC     4.400714  4.508124  2.648095 -3.360090  
           WMT     4.835141  4.866224  3.398263 -0.703873  
           XOM     5.032413  5.078663  2.582762  2.399187  
           XYZ     4.109759  4.291963 -0.541365  0.056086  

[5662 rows x 8 columns]

### 4. Calculate Monthly Returns for different time horizons as features.

- To capture time series dynamics that reflect, for example, momentum patterns, we compute historical returns using the method .pct_change(lag). returns over various monthly periods as identified by lags.

In [51]:
def calculate_returns(df):

    outlier_cutoff = 0.005

    lags = [1,2,3,6,9,12]

    for lag in lags:
        df[f'return_{lag}m'] = (df['adj close']
                            .pct_change(lag)
                           .pipe(lambda x: x.clip(lower=x.quantile(outlier_cutoff),
                                                  upper=x.quantile(1-outlier_cutoff)))
                            .add(1)
                            .pow(1/lag)
                            .sub(1))
    return df

data = data.groupby(level=1, group_keys=False).apply(calculate_returns).dropna()

data



adj close  garman_klass_vol        rsi    bb_low  \
date       ticker                                                      
2024-03-31 AAPL    169.933456          8.246013  41.926509  5.115188   
           ABBV    169.996918          8.455500  62.641755  5.109555   
           ABT     109.338379          6.828631  47.639791  4.654295   
           ACN     337.552582         10.895009  41.692945  5.777120   
           ADBE    504.600006         12.592603  38.587372  6.172837   
...                       ...               ...        ...       ...   
2026-03-31 WDAY    133.380005          7.334381  35.589722  4.841546   
           WFC      76.389999          5.556414  35.566775  4.293305   
           WMT     120.841995          7.075625  45.694634  4.804058   
           XOM     158.160004          7.951514  65.453187  4.986162   
           XYZ      58.990002          4.753294  47.622919  3.927554   

                     bb_mid   bb_high       atr      macd  return_1m  \
date       ticker                                                      
2024-03-31 AAPL    5.148172  5.181157  0.068348 -1.163779  -0.051286   
           ABBV    5.126106  5.142657 -0.342560  0.386982   0.034365   
           ABT     4.723874  4.793453 -0.108879 -0.989671  -0.041976   
           ACN     5.880985  5.984850  1.239836 -1.598348  -0.075164   
           ADBE    6.282639  6.392441  1.317077 -1.881835  -0.099379   
...                     ...       ...       ...       ...        ...   
2026-03-31 WDAY    4.930753  5.019961  0.253458 -1.366942  -0.002841   
           WFC     4.400714  4.508124  2.648095 -3.360090  -0.062124   
           WMT     4.835141  4.866224  3.398263 -0.703873  -0.053615   
           XOM     5.032413  5.078663  2.582762  2.399187   0.037115   
           XYZ     4.109759  4.291963 -0.541365  0.056086  -0.073940   

                   return_2m  return_3m  return_6m  return_9m  return_12m  
date       ticker                                                          
2024-03-31 AAPL    -0.035054  -0.037451   0.000695  -0.013168    0.003711  
           ABBV     0.052456   0.058597   0.037291   0.037552    0.014519  
           ABT      0.002251   0.012381   0.028805   0.006332    0.011367  
           ACN     -0.024019  -0.002891   0.021695   0.014282    0.017069  
           ADBE    -0.096233  -0.054297  -0.001740   0.003498    0.022063  
...                      ...        ...        ...        ...         ...  
2026-03-31 WDAY    -0.128543  -0.146836  -0.092688  -0.063186   -0.045603  
           WFC     -0.076197  -0.059832  -0.011896  -0.001557    0.006989  
           WMT      0.008150   0.028162   0.027583   0.024539    0.027742  
           XOM      0.061124   0.097802   0.059440   0.046370    0.026927  
           XYZ     -0.011986  -0.032269  -0.033274  -0.015557    0.006881  

[3461 rows x 14 columns]

### 5. Download Fama-French Factors and Calculate Rolling Factor Betas.

- We will introduce the Fama-French data to estimate the exposure of assets to common risk factors using linear regressionn.
- The five Fama-French factors, namely market risk, size, value, operating profitability, and investment have been shown empirically to explain asset returns and are commonly used to assess the risk/return profile of portfolios. Hence, it is natural to include past factor exposures as financial features in models.
- We can access the historical factor returns using the pandas-datareader and estimate historical exposures using the RollingOLS rolling linear regression.

In [60]:
import zipfile, io, requests

url = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_CSV.zip"

r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))

ff_factors = pd.read_csv(
    z.open(z.namelist()[0]),
    skiprows=3,
    index_col=0
)

# The file has annual data appended at the bottom, cut it off
ff_factors = ff_factors[ff_factors.index.astype(str).str.strip().str.match(r'^\d{6}$')]

# Convert index to datetime to match your data's date index
ff_factors.index = pd.to_datetime(ff_factors.index, format='%Y%m') + pd.offsets.MonthEnd(0)

# Filter from 2010 onwards
ff_factors = ff_factors.loc['2010':]

# drop RF column
factor_data = ff_factors.apply(pd.to_numeric, errors='coerce').drop('RF', axis=1)

factor_data.index.name = 'date'

factor_data

,Mkt-RF,SMB,HML,RMW,CMA
date,,,,,
2010-01-31,-3.35,0.40,0.33,-1.08,0.51
2010-02-28,3.39,1.49,3.18,-0.29,1.42
2010-03-31,6.30,1.83,2.19,-0.61,1.74
2010-04-30,2.00,4.96,2.96,0.61,1.75
2010-05-31,-7.90,0.08,-2.48,1.30,-0.24
...,...,...,...,...,...
2025-09-30,3.39,-2.18,-1.05,-2.06,-2.22
2025-10-31,1.96,-1.30,-3.10,-5.21,-4.03
2025-11-30,-0.13,1.47,3.76,1.42,0.68
